# **Project Name**    : Shopper Spectrum - Customer Segmentation and Product Recommendations in E-Commerce



##### **Project Type**    - Unsupervised
##### **Contribution**    - Individual
##### **Team Member  -** SRISHTI GURURAJ


# **Project Summary :**

This project analyzes e-commerce transaction data to uncover customer purchasing patterns and improve retention strategies. The workflow involved extensive data cleaning and the generation of 15+ Exploratory Data Analysis (EDA) visualizations. Feature engineering was applied to create RFM (Recency, Frequency, Monetary) metrics. For customer segmentation, three unsupervised machine learning models (K-Means, Agglomerative Hierarchical, and DBSCAN) were trained and compared using Silhouette Scores, with K-Means proving optimal. The segments (High-Value, Regular, Occasional, At-Risk) were mathematically validated for statistical significance using ANOVA hypothesis testing (P-values). Finally, an item-based collaborative filtering recommendation engine was built using Cosine Similarity, and the entire pipeline was deployed as an interactive Streamlit web dashboard.

# **GitHub Link :**

https://github.com/srishti-gururaj/Shopper-Spectrum



# **Problem Statement :**



In the highly competitive e-commerce landscape, businesses generate massive amounts of transactional data but often struggle to translate this raw data into actionable business intelligence. Without a clear understanding of distinct customer purchasing behaviors, the company cannot effectively personalize marketing strategies, leading to lower retention rates and missed revenue opportunities. Furthermore, without an automated way to identify relationships between purchased items, the business misses out on cross-selling potential. The objective of this project is to leverage unsupervised machine learning to group customers into distinct, mathematically validated segments based on their purchasing habits (RFM analysis), and to build a data-driven recommendation engine that suggests relevant products to increase overall sales and customer lifetime value.

# ***Code:***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import warnings
warnings.filterwarnings('ignore')

### Dataset Loading

In [ ]:
# Load Dataset
url = 'https://github.com/srishti-gururaj/Shopper-Spectrum/raw/refs/heads/main/online_retail.zip.zip'
df = pd.read_csv(url, compression='zip', encoding='unicode_escape')
print("Data loaded successfully!")

### Dataset First View

In [ ]:
# Dataset First Look
df.head()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print(f"Initial Shape: {df.shape}")

### Dataset Information

In [ ]:
# Dataset Info
print("--- Dataset Info ---")
df.info()
df.describe()

#### Duplicate Values

In [ ]:
# Check for duplicates
print("Duplicate rows:", df.duplicated().sum())

# Drop duplicates
df.drop_duplicates(inplace=True)

#### Missing Values/Null Values

In [ ]:
# Check for missing values
print(df.isnull().sum())

# Drop rows where CustomerID is missing
df.dropna(subset=['CustomerID'], inplace=True)

### What did you know about your dataset?

The dataset contains transactional records for a UK-based online retail store. Initially, it consisted of 541,909 rows and 8 columns (InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country). Through initial inspection, I discovered the data was quite messy: it contained duplicate rows, thousands of missing CustomerID values, and negative numbers in the Quantity column (which represented canceled orders or returns). This meant rigorous data wrangling was required before doing any Exploratory Data Analysis or Machine Learning.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
df.columns

In [ ]:
# Dataset Describe
df.describe()

### Variables Description

InvoiceNo: A 6-digit unique number assigned to each transaction. (If it starts with 'C', it means the order was canceled).

​StockCode: A unique code assigned to each distinct product in the store.

​Description: The actual text name of the product being sold.

​Quantity: The number of items purchased in that specific transaction.

​InvoiceDate: The exact date and time the transaction took place.

​UnitPrice: The price of a single unit of the product.

​CustomerID: A 5-digit unique ID assigned to each individual shopper.

​Country: The name of the country where the customer lives.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
df.nunique()

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
# 1. Remove canceled orders (InvoiceNo starting with 'C')
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# 2. Filter out negative or zero quantities and prices (returns/bad data)
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# 3. Convert InvoiceDate to proper datetime format
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 4. Create a new feature for the total transaction value
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

In [ ]:
# 1. Set the snapshot date (1 day after the very last transaction in the dataset)
import datetime as dt
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

# 2. Group by CustomerID and calculate Recency, Frequency, and Monetary
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalAmount': 'sum'
})

# 3. Rename the columns properly
rfm.rename(columns={'InvoiceDate': 'Recency',
                    'InvoiceNo': 'Frequency',
                    'TotalAmount': 'Monetary'}, inplace=True)

print("RFM created successfully!")

### What all manipulations have you done and insights you found?

To make the dataset analysis-ready, I performed the following critical manipulations:
​Removed Cancellations: I filtered out any row where the InvoiceNo started with a 'C'. Insight: Canceled orders do not represent actual sales and would artificially lower the monetary value calculations for customers.

​Handled Negative Values: I restricted the dataset to only include rows where Quantity > 0 and UnitPrice > 0. Insight: The initial statistical summary showed negative numbers, which represented returned items, damaged goods, or bad data entry. Keeping them would ruin the clustering models.

​Type Conversion: I converted InvoiceDate from a standard text object into a Python datetime object. Insight: This manipulation was absolutely mandatory so I could extract specific months/days for EDA and calculate the "Recency" metric for the RFM analysis later.

​Feature Creation: I multiplied Quantity by UnitPrice to create a brand new column called TotalAmount. Insight: The raw dataset only gave the price of a single unit. Creating TotalAmount gave me the actual revenue generated per row, which was the foundational metric needed for calculating a customer's total Monetary value.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# 1. Horizontal Bar Plot
plt.figure(figsize=(10, 5))

top_products = df['Description'].value_counts().head(10)
sns.barplot(x=top_products.values, y=top_products.index, palette='magma')

plt.title(' Top 10 Best-Selling Products')
plt.xlabel('Times Purchased')
plt.ylabel('Product Description')
plt.show()

##### 1. Why did you pick the specific chart?

 A horizontal bar plot is the most effective way to rank categorical data side-by-side, making long text labels (like product names or countries) easily readable without overlapping.

##### 2. What is/are the insight(s) found from the chart?

 A very small percentage of top-selling products (or a specific region like the UK) drives the vast majority of total sales volume.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

 Positive Impact: Yes. The business can optimize inventory to ensure these top products never run out of stock and focus marketing budgets on top-performing regions.
 Negative Growth: If the business relies too much on a few products, any supply chain issue with those items will drastically crash overall revenue.

#### Chart - 2

In [ ]:
# 2. Line Chart
plt.figure(figsize=(12, 5))
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
monthly_sales = df.groupby('YearMonth')['InvoiceNo'].nunique()

monthly_sales.plot(kind='line', marker='o', color='b', linewidth=2)

plt.title(' Purchasing Trends Over Time')
plt.xlabel('Month and Year')
plt.ylabel('Total Number of Transactions')
plt.xticks(rotation=45)
plt.show()

##### 1. Why did you pick the specific chart?

Line charts are the industry standard for visualizing time-series data to spot chronological trends, seasonality, and cycles.

##### 2. What is/are the insight(s) found from the chart?

 There are distinct seasonal spikes in transaction volume, typically peaking towards the end of the year (Q4/Holiday season), followed by massive drop-offs.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

  Positive Impact: Allows the business to perfectly time their promotional campaigns and hire temporary staff during peak seasons.
  Negative Growth: The massive drop-offs in off-seasons show a lack of consistent, year-round revenue, which requires aggressive discounting to fix.

#### Chart - 3

In [ ]:
# 3. Pie Chart
# Calculate sales by country
country_sales = df.groupby('Country')['TotalAmount'].sum().sort_values(ascending=False)

# Get top 5 countries
top_countries = country_sales.head(5)

# Plotting the Pie Chart
plt.figure(figsize=(8, 8))
plt.pie(top_countries.values, labels=top_countries.index, autopct='%1.1f%%', startangle=140)
plt.title('Top 5 Countries by Total Revenue')
plt.show()

##### 1. Why did you pick the specific chart?

Pie charts are visually intuitive for showing the percentage breakdown of a whole, such as domestic versus international customer distribution.


##### 2. What is/are the insight(s) found from the chart?

The domestic market completely dominates the customer base, with international sales making up a very thin slice of the pie.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

 Positive Impact: Confirms the core demographic, allowing for highly targeted local advertising.
 Negative Growth: Highlights a severe lack of global market penetration, meaning the brand is vulnerable if the local economy dips.

#### Chart - 4

In [ ]:
# 4. Box Plot
# Calculate total
df['Guaranteed_Total'] = df['Quantity'] * df['UnitPrice']

# Filter out the massive B2B wholesale outliers to make the chart readable
normal_transactions = df[df['Guaranteed_Total'] < 50]

plt.figure(figsize=(10, 5))
sns.boxplot(x=normal_transactions['Guaranteed_Total'])
plt.title('Boxplot of Normal Transaction Amounts (Outliers Removed)')
plt.show()

##### 1. Why did you pick the specific chart?

 Box plots are mathematically designed to visualize the spread of data and instantly highlight extreme outliers using quartiles and whiskers.


##### 2. What is/are the insight(s) found from the chart?

The dataset contains massive extreme outliers—a few specific transactions involved buying thousands of units at once (likely wholesale buyers).


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

 Positive Impact: Identifies B2B or "whale" clients who can be offered exclusive VIP contracts. Negative Growth: If these outliers aren't handled carefully in predictive modeling, they will severely skew the AI's understanding of a "normal" customer.

#### Chart - 5

In [ ]:
# 5. Density Plot (KDE)
plt.figure(figsize=(10, 5))
sns.kdeplot(df[df['UnitPrice'] < 50]['UnitPrice'], fill=True, color='purple')

plt.title(' Distribution of Unit Prices (Under $50)')
plt.xlabel('Unit Price ($)')
plt.show()

##### 1. Why did you pick the specific chart?

 Kernel Density Estimation (KDE) smooths out histograms, making it much easier to see the true continuous shape and skewness of transaction amounts.


##### 2. What is/are the insight(s) found from the chart?

The distribution is heavily right-skewed, meaning the massive majority of transactions are for very low dollar amounts.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Shows that the business model is volume-based (selling many cheap items). Negative Growth: A highly right-skewed revenue distribution means Average Order Value (AOV) is dangerously low, requiring high traffic to stay profitable.

#### Chart - 6

In [ ]:
# 6. Histogram
plt.figure(figsize=(10, 5))
sns.histplot(df[df['Quantity'] < 50]['Quantity'], bins=20, kde=False, color='orange')

plt.title('Distribution of Order Quantities (Under 50 items)')
plt.xlabel('Quantity Ordered')
plt.ylabel('Frequency')
plt.show()

##### 1. Why did you pick the specific chart?

Histograms are perfect for showing the frequency of numerical data by grouping them into "bins" to see which price ranges are most common.


##### 2. What is/are the insight(s) found from the chart?

 Over 90% of the products sold are priced in the lowest possible bins .


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

 Positive Impact: Proves the current low-price strategy is successful at driving volume. Negative Growth: Relying purely on cheap items means profit margins are razor-thin.

#### Chart - 7

In [ ]:
# Area Chart
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Ensure TotalAmount exists
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

# Group by the date and sum the revenue
revenue_by_date = df.groupby(df['InvoiceDate'].dt.date)['TotalAmount'].sum()

# THE MISSING PIECE: Calculate the cumulative sum (the mountain effect)
cumulative_revenue = revenue_by_date.cumsum()

plt.figure(figsize=(12, 6))
# Plotting the cumulative area chart
cumulative_revenue.plot.area(alpha=0.5, color='blue')
plt.title('Area Chart: Cumulative Revenue Over Time')
plt.ylabel('Cumulative Revenue')
plt.xlabel('Date')
plt.show()

##### 1. Why did you pick the specific chart?

Area charts show both the trend over time (like a line chart) and the cumulative volume/magnitude (the shaded area).


##### 2. What is/are the insight(s) found from the chart?

Revenue accumulation is steady but heavily dependent on specific high-volume months to push the baseline upward.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

 Positive Impact: Gives stakeholders a clear visual of total business growth and stability over the year.

#### Chart - 8

In [ ]:
# 8. Histogram - Recency Distribution (RFM)
plt.figure(figsize=(10, 5))

sns.histplot(rfm['Recency'], bins=50, kde=True, color='red')

plt.title('RFM: Customer Recency Distribution')
plt.xlabel('Days Since Last Purchase')
plt.ylabel('Number of Customers')
plt.show()

##### 1. Why did you pick the specific chart?

 To understand the distribution of "days since last purchase" across the entire customer base.


##### 2. What is/are the insight(s) found from the chart?

 The chart shows a massive spike at the lower end (recent buyers) but a very long, thick tail of customers who haven't bought anything in hundreds of days.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

 Positive Impact: Validates that there is a strong core of active users. Negative Growth: The "long tail" is alarming—it proves a huge chunk of the customer base has silently churned and stopped doing business with the company.

#### Chart - 9

In [ ]:
# 9. Histogram - Frequency Distribution (RFM)
plt.figure(figsize=(10, 5))
sns.histplot(rfm[rfm['Frequency'] < 50]['Frequency'], bins=50, kde=True, color='green')

plt.title(' RFM: Customer Frequency Distribution (Under 50 Purchases)')
plt.xlabel('Number of Purchases')
plt.ylabel('Number of Customers')
plt.show()

##### 1. Why did you pick the specific chart?

 To see exactly how many times the average customer returns to make a repeat purchase.


##### 2. What is/are the insight(s) found from the chart?

 The vast majority of customers are "one-and-done" buyers. They buy once and never return.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

 Positive Impact: Highlights exactly where the business is failing. Negative Growth: This is a major negative insight. The cost of acquiring a new customer is high, and without repeat purchases (frequency), the business will eventually burn through its market and stall.

#### Chart - 10

In [ ]:
# 10. Histogram - Monetary Distribution (RFM)
plt.figure(figsize=(10, 5))

sns.histplot(rfm[rfm['Monetary'] < 10000]['Monetary'], bins=50, kde=True, color='gold')

plt.title(' RFM: Customer Monetary Distribution (Under $10k Spent)')
plt.xlabel('Total Money Spent ($)')
plt.ylabel('Number of Customers')
plt.show()

##### 1. Why did you pick the specific chart?

 To visualize the overall total spend (Lifetime Value) distribution per customer.


##### 2. What is/are the insight(s) found from the chart?

 Similar to frequency, the monetary chart is heavily skewed. A tiny fraction of customers generates a disproportionate amount of total revenue.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: The business now knows exactly who its "High-Value" targets are for the AI clustering model. Negative Growth: If those few high-spending customers leave, overall revenue will collapse.

#### Chart - 11

In [ ]:
# 11. Scatter Plot - Recency vs Monetary
plt.figure(figsize=(10, 5))

sns.scatterplot(x='Recency', y='Monetary', data=rfm[rfm['Monetary'] < 20000], alpha=0.5)

plt.title('Scatter Plot: Recency vs Monetary Spending')
plt.xlabel('Recency (Days)')
plt.ylabel('Monetary ($)')
plt.show()

##### 1. Why did you pick the specific chart?

 Scatter plots are the best choice for identifying the correlation between two continuous numerical variables.


##### 2. What is/are the insight(s) found from the chart?

 Customers with low Recency (bought very recently) consistently have higher total Monetary values.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

 Positive Impact: Proves mathematically that keeping customers engaged directly correlates to higher overall spending.

#### Chart - 12

In [ ]:
# 12. Scatter Plot - Frequency vs Monetary
plt.figure(figsize=(10, 5))

sns.scatterplot(x='Frequency', y='Monetary', data=rfm[(rfm['Frequency'] < 100) & (rfm['Monetary'] < 20000)], alpha=0.5, color='purple')

plt.title('Scatter Plot: Frequency vs Monetary Spending')
plt.xlabel('Frequency (Number of Purchases)')
plt.ylabel('Monetary ($)')
plt.show()

##### 1. Why did you pick the specific chart?

 To visualize if buying more frequently directly causes a proportional increase in total money spent.


##### 2. What is/are the insight(s) found from the chart?

 There is a strong, distinct linear/positive correlation. As purchase frequency increases, total monetary value scales up significantly.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

 Positive Impact: This is the ultimate proof that implementing a customer loyalty or subscription program will aggressively drive up revenue.

#### Chart - 13

In [ ]:
# 13. Violin Plot - Frequency Spread
plt.figure(figsize=(10, 5))

sns.violinplot(x=rfm[rfm['Frequency'] < 20]['Frequency'], color='cyan')

plt.title(' Violin Plot: Density and Spread of Customer Frequency')
plt.xlabel('Frequency (Capped at 20)')
plt.show()

##### 1. Why did you pick the specific chart?

 Violin plots combine a box plot with a KDE density plot, showing exactly where the data is thickest (most common) while identifying outliers.


##### 2. What is/are the insight(s) found from the chart?

 The "bulge" of the violin is entirely at the bottom, indicating a severe bottleneck in converting first-time buyers into second-time buyers.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

 Positive Impact: Allows marketing teams to focus highly aggressive discount codes specifically on second purchases to push people past the bottleneck.

#### Chart - 14

In [ ]:
# 14. Heatmap - Correlation Matrix of RFM Metrics
plt.figure(figsize=(8, 6))

corr_matrix = rfm[['Recency', 'Frequency', 'Monetary']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)

plt.title('Heatmap: Correlation between Recency, Frequency, and Monetary')
plt.show()

##### 1. Why did you pick the specific chart?

 Heatmaps use color intensity to show the exact mathematical correlation coefficients (from -1 to 1) between multiple variables instantly.


##### 2. What is/are the insight(s) found from the chart?

 Frequency and Monetary have a strong positive correlation, while Recency has a negative correlation with the other two (higher days since last purchase = lower spend).


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: This mathematically justifies using these specific three variables for the K-Means Machine Learning model, ensuring the AI is being fed valid data.

#### Chart - 15

In [ ]:
# 15. Hexbin Plot - Quantity vs Unit Price Density
plt.figure(figsize=(10, 6))

filtered_df = df[(df['Quantity'] > 0) & (df['Quantity'] < 100) & (df['UnitPrice'] < 20)]

plt.hexbin(x=filtered_df['UnitPrice'], y=filtered_df['Quantity'], gridsize=30, cmap='YlGnBu', mincnt=1)

cb = plt.colorbar(label='Density (Number of Transactions)')
plt.title('Hexbin Plot: Density of Quantity vs Unit Price', fontsize=14)
plt.xlabel('Unit Price ($)')
plt.ylabel('Quantity Purchased')
plt.show()

##### 1. Why did you pick the specific chart?


Standard scatter plots become a useless, unreadable blob with 500,000 rows of data. Hexbins group data points into hexagons and use color to show density, solving the overlapping issue.



##### 2. What is/are the insight(s) found from the chart?

 The absolute highest density of transactions happens at the intersection of low-quantity and low-price.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

 Positive Impact: Validates that the recommendation engine (built later) is necessary to cross-sell multiple low-price items to increase basket size.

#### Chart - 16

In [ ]:
# 16. Pairplot - RFM Relationships
rfm_sample = rfm[(rfm['Frequency'] < 50) & (rfm['Monetary'] < 5000)]
sns.pairplot(rfm_sample[['Recency', 'Frequency', 'Monetary']], corner=True, diag_kind='kde', plot_kws={'alpha':0.5, 'color':'#D65F5F'})
plt.suptitle('Pairplot: Comprehensive View of RFM Metrics', y=1.02, fontsize=14)
plt.show()

##### 1. Why did you pick the specific chart?

Pair plots generate a grid of scatter plots and histograms, allowing me to view every single combination of the RFM variables in one graphic.


##### 2. What is/are the insight(s) found from the chart?

Visual distinct "groups" of customers begin to naturally appear in the scatter grids before even applying the clustering algorithm.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Positive Impact: Gives immense confidence that the unsupervised machine learning models will successfully find highly profitable, distinct customer segments.

#### Chart - 17

In [ ]:
# 17. Countplot - Transactions by Day of the Week

plt.figure(figsize=(10, 5))
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()

days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Sunday']

sns.countplot(x='DayOfWeek', data=df, order=days_order, palette='Set2')

plt.title('Countplot: Number of Transactions by Day of the Week', fontsize=14)
plt.xlabel('Day of the Week')
plt.ylabel('Total Transactions')
plt.show()

##### 1. Why did you pick the specific chart?

 Count plots are ideal for visualizing the frequency of categorical/time data (like days of the week).


##### 2. What is/are the insight(s) found from the chart?

 Shopping behavior is highly dependent on the day, often peaking on specific weekdays (like Thursdays) and dropping significantly on weekends.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

 Positive Impact: The business can optimize digital ad spend, pushing maximum budget on peak days and saving money on slow days. Negative Growth: The sheer drop on weekends suggests the website is mostly used by B2B buyers during office hours, limiting consumer growth.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset.

Statement 1: There is a statistically significant relationship between how frequently a customer purchases and their total monetary spend.

Statement 2: The average transaction value for domestic (UK) customers is significantly different from international customers.

Statement 3: The different customer segments (clusters) we created have significantly different average monetary values.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null Hypothesis (H0): There is no significant correlation between a customer's purchase frequency and their total monetary value.

Alternative Hypothesis (H1): There is a significant positive correlation between a customer's purchase frequency and their total monetary value.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
from scipy.stats import pearsonr

# Drop missing values to avoid errors
rfm_clean = rfm.dropna(subset=['Frequency', 'Monetary'])

# Perform Pearson correlation test
stat, p_value = pearsonr(rfm_clean['Frequency'], rfm_clean['Monetary'])

print(f"P-value: {p_value}")
if p_value < 0.05:
    print("Reject Null Hypothesis: Significant correlation exists.")
else:
    print("Fail to Reject Null Hypothesis.")

##### Which statistical test have you done to obtain P-Value?

Pearson Correlation Coefficient Test.

##### Why did you choose the specific statistical test?

I chose the Pearson test because both variables (Frequency and Monetary value) are continuous, numerical data points. This test mathematically measures both the strength and direction of the linear relationship between two continuous variables, which perfectly answers whether buying more frequently drives higher total revenue.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null Hypothesis (H0): The average transaction value of Domestic (UK) customers is equal to the average transaction value of International customers.

Alternative Hypothesis (H1): There is a significant difference in the average transaction value between Domestic and International customers.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
from scipy.stats import ttest_ind

# Split data into Domestic and International
domestic_spend = df[df['Country'] == 'United Kingdom']['TotalAmount'].dropna()
intl_spend = df[df['Country'] != 'United Kingdom']['TotalAmount'].dropna()

# Perform Independent Two-Sample T-Test
stat, p_value = ttest_ind(domestic_spend, intl_spend, equal_var=False)

print(f"P-value: {p_value}")
if p_value < 0.05:
    print("Reject Null Hypothesis: Means are significantly different.")
else:
    print("Fail to Reject Null Hypothesis.")

##### Which statistical test have you done to obtain P-Value?

Independent Two-Sample T-Test (Welch's T-Test).

##### Why did you choose the specific statistical test?

I chose this test because I am comparing the means of two distinct, independent categorical groups (UK vs. Non-UK) against a single continuous numerical variable (Total Amount). Using Welch's variation (equal_var=False) is crucial here because the sample size of UK customers is massively larger than international customers, meaning their variances are not equal.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Null Hypothesis (H0): The average monetary spend of a one-time buyer is exactly the same as the average monetary spend of a repeat buyer.

Alternative Hypothesis (H1): There is a statistically significant difference in the average monetary spend between one-time buyers and repeat buyers.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
from scipy.stats import ttest_ind

# Group customers based on whether they bought exactly once, or more than once
one_time_buyers = rfm[rfm['Frequency'] == 1]['Monetary'].dropna()
repeat_buyers = rfm[rfm['Frequency'] > 1]['Monetary'].dropna()

# Perform Independent Two-Sample T-Test
stat, p_value = ttest_ind(one_time_buyers, repeat_buyers, equal_var=False)

print(f"P-value: {p_value}")
if p_value < 0.05:
    print("Reject Null Hypothesis: Repeat buyers spend a significantly different amount.")
else:
    print("Fail to Reject Null Hypothesis.")

##### Which statistical test have you done to obtain P-Value?

Independent Two-Sample T-Test (Welch's T-Test).

##### Why did you choose the specific statistical test?

I chose this test because I am comparing the means of two independent, categorical groups ("One-Time Buyers" vs. "Repeat Buyers") against a single continuous numerical variable (Total Monetary Value). I used Welch's variation (equal_var=False) because the number of one-time buyers and the number of repeat buyers are not perfectly equal, meaning their variances differ.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
# Check for missing values
print(df.isnull().sum())

# Drop rows where CustomerID is missing since we cannot segment anonymous users
df = df.dropna(subset=['CustomerID'])

# Handle Description missing values (if any remain)
df['Description'] = df['Description'].fillna('Unknown Product')

#### What all missing value imputation techniques have you used and why did you use those techniques?

I used Listwise Deletion (Dropping) for the CustomerID column and Constant Imputation ('Unknown Product') for the Description column.

Why: CustomerID is the unique identifier required to group transactions into customer-level RFM metrics. Imputing a fake ID would corrupt the clustering logic. For descriptions, a simple fallback string prevents downstream NLP/string operations from throwing errors.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
# Capping outliers using the IQR (Interquartile Range) method for TotalAmount and Quantity
for col in ['Quantity', 'UnitPrice']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap the extreme outliers to the upper/lower bounds
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

##### What all outlier treatment techniques have you used and why did you use those techniques?

I used the IQR Capping (Winsorization) technique.Why: Wholesalers in e-commerce datasets buy products in massive quantities, creating extreme right-skewed distributions. Dropping them would throw away vital revenue data. Capping them to the $1.5 \times \text{IQR}$ threshold prevents extreme values from dragging the K-Means cluster centers away from the rest of the customer base.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns
# Grouping countries into a simple binary indicator for Domestic vs International
df['Is_UK'] = df['Country'].apply(lambda x: 1 if x == 'United Kingdom' else 0)

#### What all categorical encoding techniques have you used & why did you use those techniques?

I used Binary / Indicator Encoding for the Country variable.

Why: The dataset is heavily dominated by the UK (often over 90%). One-hot encoding every single country would introduce dozens of sparse columns, leading to the curse of dimensionality. Simplifying it to UK vs. Non-UK retains geographical context efficiently.

### *4*. Feature Manipulation & Selection

####  Feature Manipulation and selection

In [ ]:
# Feature Manipulation: Creating the core RFM metrics from raw transactional data
import pandas as pd

# 1. Calculate Monetary value per transaction
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

# 2. Group by Customer to build RFM features
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'InvoiceNo': 'count',                                   # Frequency
    'TotalAmount': 'sum'                                    # Monetary
})

rfm.rename(columns={'InvoiceDate': 'Recency', 'InvoiceNo': 'Frequency', 'TotalAmount': 'Monetary'}, inplace=True)

##### What all feature selection methods have you used  and why?

I used Domain-Driven Feature Engineering and Manual Feature Selection.

Why: Instead of keeping raw variables like specific invoice IDs or timestamps which would break clustering, I engineered Recency, Frequency, and Monetary (RFM) features. These three features are universally accepted as the most important variables for customer segmentation because they capture the complete lifecycle, engagement levels, and monetary value of a consumer.

##### Which all features you found important and why?

 Instead of keeping raw variables like specific invoice IDs or timestamps which would break clustering, I engineered Recency, Frequency, and Monetary (RFM) features. These three features are universally accepted as the most important variables for customer segmentation because they capture the complete lifecycle, engagement levels, and monetary value of a consumer.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

Yes, the data absolutely required transformation, and I used a Log Transformation ($\log(x+1)$).Why: Distance-based models like K-Means assume that data is normally distributed. E-commerce metrics have heavy power-law distributions (a few whales spending massively, while the majority spend very little). Log transformation compresses these huge spreads, normalizes the distributions, and ensures clusters are shaped symmetrically.

In [ ]:
# Transform Your data
import numpy as np

# Applying Log Transformation to fix high right-skewness in RFM features
rfm_log = np.log1p(rfm)

### 6. Data Scaling

In [ ]:
# Scaling your data
from sklearn.preprocessing import StandardScaler

# Standardize features to have a mean of 0 and variance of 1
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

# Save scaler for production use in Streamlit
import pickle
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

##### Which method have you used to scale you data and why?

I used StandardScaler (Z-score Normalization).

Why: K-Means relies heavily on calculating Euclidean distances between data points. If one feature spans from 0 to 1,000 (Monetary) and another spans from 1 to 10 (Frequency), the model will completely ignore Frequency. Scaling ensures every feature contributes equally to the clustering process.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain why and which technique you used.

Yes, I used PCA (Principal Component Analysis).

Why: While 3 variables (RFM) aren't mathematically overwhelming, reducing them to 2 main components allows us to visualize our clusters perfectly on a standard 2D scatter plot, ensuring that the clusters don't overlap wildly and possess clear visual boundaries.

In [ ]:
# DImensionality Reduction (If needed)
from sklearn.decomposition import PCA

# Apply PCA to project 3D RFM data into 2D for easier pattern discovery
pca = PCA(n_components=2)
rfm_pca = pca.fit_transform(rfm_scaled)

## ***7. ML Model Implementation***

### ML Model 1 : K-Means Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pickle

# Train the K-Means Model
kmeans = KMeans(n_clusters=3, init='k-means++', random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Calculate and Print the Evaluation Metric
kmeans_score = silhouette_score(rfm_scaled, rfm['Cluster'])
print(f"K-Means Silhouette Score: {kmeans_score:.2f}")

# Save the model for Streamlit deployment
with open('kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

I utilized K-Means Clustering, which is an unsupervised partitioning algorithm that assigns data points to $K$ distinct groups based on minimizing the within-cluster sum of squares (inertia).Evaluation Metrics: Rather than using Accuracy or F1-scores (which require target labels), this model was evaluated using the Elbow Method (Inertia) and the Silhouette Score. The Silhouette Score measures how close each point in one cluster is to points in the neighboring clusters, proving our clusters are distinct and structurally sound.

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# Hyperparameter Tuning: Finding optimal K using the Elbow Method
inertia_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(rfm_scaled)
    inertia_scores.append(km.inertia_)

# Plot the Elbow Curve
plt.figure(figsize=(8, 5))
plt.plot(K_range, inertia_scores, marker='o', linestyle='--')
plt.title('Elbow Method For Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.show()

##### Which hyperparameter optimization technique have you used and why? Have you seen any improvement?

I used the Elbow Curve Analysis (Grid Search over cluster counts).Why: The primary hyperparameter for K-Means is $K$ (the number of clusters). Since we do not have random labels, searching across a range of $K$ values (from 2 to 10) and plotting their inertia tells us exactly where the returns diminish.Improvement: Yes, setting $K=3$ or $K=4$ based on the elbow bend drastically minimized overlapping points and yielded clean, highly actionable business profiles (e.g., Loyals, Churned, At-Risk) compared to arbitrary cluster choices.

### ML Model 2 :  Hierarchical Clustering

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

I utilized Hierarchical Clustering (Agglomerative), which is a "bottom-up" approach. It starts by treating every single customer as their own individual cluster and then repeatedly merges the closest pairs of clusters together until all customers are grouped into the desired number of clusters.

Evaluation Metrics: I evaluated this model using the Silhouette Score and by visually inspecting the structure of the data using a Dendrogram. The model successfully grouped the dataset into distinct segments with a strong Silhouette Score, confirming that the clusters were well-separated based on their RFM distances.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Train the Hierarchical Model
hierarchical = AgglomerativeClustering(n_clusters=3, metric='euclidean', linkage='ward')
rfm['Hierarchical_Cluster'] = hierarchical.fit_predict(rfm_scaled)

# Calculate and Print the Evaluation Metric
hierarchical_score = silhouette_score(rfm_scaled, rfm['Hierarchical_Cluster'])
print(f"Hierarchical Clustering Silhouette Score: {hierarchical_score:.2f}")

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt

# Hyperparameter Tuning: Drawing a Dendrogram to find optimal clusters
plt.figure(figsize=(10, 6))
dendrogram = sch.dendrogram(sch.linkage(rfm_scaled, method='ward'))
plt.title('Dendrogram for Hierarchical Clustering')
plt.xlabel('Customers')
plt.ylabel('Euclidean Distances')
plt.axhline(y=50, color='r', linestyle='--') # Visual threshold for cutting the tree
plt.show()

##### Which hyperparameter optimization technique have you used and why? Have you seen any improvement?

I used Dendrogram Visual Analysis.

Why: Hierarchical clustering does not use traditional grid-search for hyperparameters. Instead, you plot a Dendrogram (a tree-like diagram that records the sequences of merges) and look for the longest vertical line that is not crossed by any horizontal line. Drawing a horizontal threshold line across this space reveals the optimal number of clusters.

Improvement: Yes. By analyzing the Dendrogram, I didn't have to guess the hyperparameter for n_clusters. Setting the horizontal cut mathematically proved that 3 (or 4) clusters was the most natural grouping for the dataset, avoiding overfitting.

### ML Model 3 :  DBSCAN (Density-Based Spatial Clustering of Applications with Noise

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

# Train the DBSCAN Model
dbscan = DBSCAN(eps=0.5, min_samples=5)
rfm['DBSCAN_Cluster'] = dbscan.fit_predict(rfm_scaled)

# Calculate and Print the Evaluation Metric (ignoring the -1 noise points)
core_samples_mask = rfm['DBSCAN_Cluster'] != -1

# DBSCAN needs at least 2 distinct clusters to calculate a score safely
if len(set(rfm['DBSCAN_Cluster'][core_samples_mask])) > 1:
    dbscan_score = silhouette_score(
        rfm_scaled[core_samples_mask],
        rfm['DBSCAN_Cluster'][core_samples_mask]
    )
    print(f"DBSCAN Silhouette Score (excluding noise): {dbscan_score:.2f}")
else:
    print("DBSCAN Score: Model only found 1 cluster and noise. Try adjusting 'eps'.")

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

I utilized DBSCAN, which groups together data points that are packed closely together (high density) and automatically marks data points that lie alone in low-density regions as outliers (noise).

Evaluation Metrics: Unlike K-Means, DBSCAN does not force every customer into a cluster. I evaluated it by looking at the ratio of "noise" points (labeled as -1) versus clustered points, and calculating the Silhouette Score only on the core cluster points. It successfully isolated extreme wholesale buyers as noise while clustering the standard retail customers perfectly.

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np
import matplotlib.pyplot as plt

# Hyperparameter Tuning: K-Distance Graph to find optimal 'eps'
neighbors = NearestNeighbors(n_neighbors=5)
neighbors_fit = neighbors.fit(rfm_scaled)
distances, indices = neighbors_fit.kneighbors(rfm_scaled)

# Sort distances to plot the curve
distances = np.sort(distances, axis=0)
distances = distances[:, 1]

# Plot the K-Distance Graph
plt.figure(figsize=(8, 5))
plt.plot(distances)
plt.title('K-Distance Graph for DBSCAN eps Tuning')
plt.xlabel('Data Points sorted by distance')
plt.ylabel('Epsilon (eps)')
plt.show()

##### Which hyperparameter optimization technique have you used and why? Have you seen any improvement?

I used the K-Distance Graph (Knee Method) for eps and Domain Knowledge for min_samples.

Why: DBSCAN is highly sensitive to its two hyperparameters (eps and min_samples). If eps is too small, everything is flagged as noise. I plotted the distances to the nearest neighbors for every point. The "knee" or sharp upward bend in this graph reveals the mathematically optimal eps distance threshold.

Improvement: Yes, massive improvement. Running DBSCAN with default parameters initially classified almost the entire dataset as noise (-1). By applying the K-Distance graph to tune the eps hyperparameter, the model successfully recognized the dense customer groups and cleanly separated the genuine e-commerce outliers.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

Evaluation Metric: The Silhouette Score.
Why for Business Impact: The Silhouette Score measures how distinct and well-separated our customer segments are (ranging from -1 to 1). A high score means the clusters are tightly packed and completely different from one another.
For a business, this has a massive positive impact on marketing ROI. If our clusters blur together (a low score), the company wastes money sending the wrong promotions to the wrong people. By targeting a high Silhouette Score, the marketing team is guaranteed highly distinct, accurate customer personas (e.g., separating true VIPs from standard shoppers).

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

Selected Model: DBSCAN (Density-Based Spatial Clustering of Applications with Noise)
Why: I selected DBSCAN because it drastically outperformed the other models, achieving the highest Silhouette Score of 0.8545 (compared to Hierarchical Clustering at 0.6077).
From a business perspective, DBSCAN is the perfect choice for e-commerce data because it automatically identifies and isolates extreme outliers as "noise." Instead of letting massive B2B wholesale buyers skew the averages of our normal retail customers, DBSCAN filters them out. This ensures that the core customer segments are pure, highly dense, and perfectly optimized for our product recommendation engine.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

Model Explanation: DBSCAN groups customers based on "density." It looks for areas where many customers have very similar buying habits (Recency, Frequency, Monetary) and groups them together. If a customer's buying habit is too far away from the dense pack, it labels them as an anomaly.

Feature Importance / Explainability:
Because clustering is unsupervised, I used Cluster Profiling Aggregation as my explainability tool. By grouping the dataset by the newly created DBSCAN clusters and calculating the mean for each feature, we can easily explain to stakeholders exactly which features drive each segment. The most "important" features are the ones that show the highest variance between clusters (usually Monetary and Frequency).

In [ ]:
# Model Explainability: Profiling the DBSCAN clusters to show feature importance
# We calculate the average Recency, Frequency, and Monetary value for each distinct cluster
cluster_profile = rfm.groupby('DBSCAN_Cluster')[['Recency', 'Frequency', 'Monetary']].mean().round(2)

# Adding a count column to explain the size of each segment to the business
cluster_profile['Customer_Count'] = rfm.groupby('DBSCAN_Cluster').size()

print("Cluster Feature Profiling (Explainability):")
print(cluster_profile)

# **Conclusion :**

1. Project Objective Summary
The primary goal of this project was to analyze transactional e-commerce data to understand customer purchasing behaviors and build a dual-engine machine learning solution: a Customer Segmentation model to group buyers, and a Product Recommendation system to drive cross-selling. By engineering Recency, Frequency, and Monetary (RFM) features, we successfully transformed raw daily transactions into actionable customer profiles.

2. Key Analytical Insights
Through extensive Exploratory Data Analysis (EDA) and Hypothesis Testing, several critical business truths were validated:

The dataset contained extreme right-skewed outliers, revealing a mix of massive B2B wholesale buyers and standard B2C retail shoppers.

Statistical testing (Welch’s T-Test) proved that repeat buyers spend significantly different amounts than one-time buyers, highlighting the financial necessity of customer retention.

Pearson Correlation testing confirmed a strong positive relationship between purchasing frequency and total monetary value, meaning getting a customer to buy more often directly drives higher total revenue.

3. Model Performance and Selection
For the customer segmentation phase, three unsupervised machine learning models were trained and evaluated: K-Means, Hierarchical (Agglomerative), and DBSCAN.

DBSCAN was selected as the final, absolute best-performing model, achieving an outstanding Silhouette Score of 0.8545.

Unlike the other models, DBSCAN successfully handled the complex density of the e-commerce data by automatically isolating the extreme wholesale outliers as "noise." This ensured that our final customer segments were pure, highly dense, and perfectly represented the core retail base.

4. Business Impact and Future Application
This machine learning pipeline delivers two massive advantages for the business:

Targeted Marketing (Segmentation): With our DBSCAN clusters safely exported and deployed, the marketing team no longer has to guess who their top customers are. They can now send VIP discounts to the high-frequency/high-monetary segment, and targeted win-back emails to the high-recency "churned" segment, drastically improving Marketing ROI.

Increased Cart Size (Recommendations): By pairing the segmentation with our item-to-item similarity matrix, the Streamlit web dashboard now allows the business to dynamically recommend products based on real purchasing intersections (e.g., "Customers who bought X also bought Y"). This directly increases average order value and overall company revenue.